In [4]:
!pip install -q transformer_lens sae_lens

In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU only")

CUDA available: True
Device: Tesla T4


In [6]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2")
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (h

Building 20 random token IDs

In [7]:
torch.manual_seed(42)

seq_len = 20
random_tokens = torch.randint(low=1000, high=10000, size=(1, seq_len))
tokens = torch.cat([random_tokens, random_tokens], dim=1).to(model.cfg.device)

print("tokens shape:", tokens.shape)
print("first half: ", tokens[0, :5].tolist())
print("second half:", tokens[0, seq_len:seq_len+5].tolist())

tokens shape: torch.Size([1, 40])
first half:  [5542, 2067, 5876, 5414, 2026]
second half: [5542, 2067, 5876, 5414, 2026]


In [8]:
with torch.no_grad():
    logits, cache = model.run_with_cache(tokens)

print("logits shape:", logits.shape)
print("cache has", len(cache), "entries") 

logits shape: torch.Size([1, 40, 50257])
cache has 210 entries


In [10]:
predicted = logits[0, -1].argmax().item()
expected = tokens[0, 20].item()

print("At position 39, model predicted token:", predicted)
print("Induction would predict token:        ", expected)
print("Induction working on this prompt?     ", predicted == expected)

At position 39, model predicted token: 5542
Induction would predict token:         5542
Induction working on this prompt?      True


In [11]:
print("Total cache entries:", len(cache))
print("-"*10)
print("First 10 keys:")
for k in list(cache.keys())[:10]:
    print(" ", k)

Total cache entries: 210
----------
First 10 keys:
  hook_embed
  hook_pos_embed
  blocks.0.hook_resid_pre
  blocks.0.ln1.hook_scale
  blocks.0.ln1.hook_normalized
  blocks.0.attn.hook_q
  blocks.0.attn.hook_k
  blocks.0.attn.hook_v
  blocks.0.attn.hook_attn_scores
  blocks.0.attn.hook_pattern


In [12]:
resid_keys = [k for k in cache.keys() if 'hook_resid' in k]
for k in resid_keys:
    print(f"{k:40s}  shape: {tuple(cache[k].shape)}")

blocks.0.hook_resid_pre                   shape: (1, 40, 768)
blocks.0.hook_resid_mid                   shape: (1, 40, 768)
blocks.0.hook_resid_post                  shape: (1, 40, 768)
blocks.1.hook_resid_pre                   shape: (1, 40, 768)
blocks.1.hook_resid_mid                   shape: (1, 40, 768)
blocks.1.hook_resid_post                  shape: (1, 40, 768)
blocks.2.hook_resid_pre                   shape: (1, 40, 768)
blocks.2.hook_resid_mid                   shape: (1, 40, 768)
blocks.2.hook_resid_post                  shape: (1, 40, 768)
blocks.3.hook_resid_pre                   shape: (1, 40, 768)
blocks.3.hook_resid_mid                   shape: (1, 40, 768)
blocks.3.hook_resid_post                  shape: (1, 40, 768)
blocks.4.hook_resid_pre                   shape: (1, 40, 768)
blocks.4.hook_resid_mid                   shape: (1, 40, 768)
blocks.4.hook_resid_post                  shape: (1, 40, 768)
blocks.5.hook_resid_pre                   shape: (1, 40, 768)
blocks.5

In [13]:
from sae_lens import SAE

sae, cfg_dict, sparsity = SAE.from_pretrained_with_cfg_and_sparsity(
    release="gpt2-small-res-jb",
    sae_id="blocks.0.hook_resid_pre",
    device="cuda",
)

cfg.json: 0.00B [00:00, ?B/s]

blocks.0.hook_resid_pre/sae_weights.safe(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

blocks.0.hook_resid_pre/sparsity.safeten(…):   0%|          | 0.00/98.4k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/sae_lens/saes/sae.py:251: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


In [14]:
print("min:", sparsity.min().item())
print("max:", sparsity.max().item())
print("mean:", sparsity.mean().item())
print("first 10 values:", sparsity[:10].tolist())

min: -10.0
max: -0.6536116600036621
mean: -4.944095134735107
first 10 values: [-3.7604904174804688, -10.0, -3.3279292583465576, -3.7754030227661133, -3.8041489124298096, -3.7332639694213867, -10.0, -3.9728732109069824, -3.790846109390259, -3.2905542850494385]


In [15]:
print("Number of dead features (log-freq <= -10):", (sparsity <= -10).sum().item())
print("Number of nearly-dead (log-freq < -8):", (sparsity < -8).sum().item())
print("Number firing > 1% of time (log-freq > -2):", (sparsity > -2).sum().item())
print("Number firing > 0.1% of time (log-freq > -3):", (sparsity > -3).sum().item())

Number of dead features (log-freq <= -10): 4488
Number of nearly-dead (log-freq < -8): 4488
Number firing > 1% of time (log-freq > -2): 126
Number firing > 0.1% of time (log-freq > -3): 319


In [16]:
resid = cache["blocks.0.hook_resid_pre"]
print("residual stream shape:", resid.shape)

with torch.no_grad():
    feature_acts = sae.encode(resid)

print("feature activations shape:", feature_acts.shape)
print("dtype:", feature_acts.dtype)
print("device:", feature_acts.device)

residual stream shape: torch.Size([1, 40, 768])
feature activations shape: torch.Size([1, 40, 24576])
dtype: torch.float32
device: cuda:0


In [17]:
nonzero_mask = feature_acts > 0
total_features = feature_acts.shape[-1]

print(f"Total features: {total_features}")
print()

print("Number of features that fired at each position:")
for pos in range(feature_acts.shape[1]):
    n_fired = nonzero_mask[0, pos].sum().item()
    print(f"  position {pos:2d}: {n_fired:4d} features fired ({100*n_fired/total_features:.1f}%)")

Total features: 24576

Number of features that fired at each position:
  position  0:  142 features fired (0.6%)
  position  1:    5 features fired (0.0%)
  position  2:    9 features fired (0.0%)
  position  3:   11 features fired (0.0%)
  position  4:   12 features fired (0.0%)
  position  5:   13 features fired (0.1%)
  position  6:   13 features fired (0.1%)
  position  7:  156 features fired (0.6%)
  position  8:   15 features fired (0.1%)
  position  9:   99 features fired (0.4%)
  position 10:   18 features fired (0.1%)
  position 11:   12 features fired (0.0%)
  position 12:   16 features fired (0.1%)
  position 13:   18 features fired (0.1%)
  position 14:   17 features fired (0.1%)
  position 15:   15 features fired (0.1%)
  position 16:   16 features fired (0.1%)
  position 17:   14 features fired (0.1%)
  position 18:   89 features fired (0.4%)
  position 19:   13 features fired (0.1%)
  position 20:   14 features fired (0.1%)
  position 21:   11 features fired (0.0%)
  pos

In [18]:
last_pos_acts = feature_acts[0, -1]   # shape (24576,)

values, indices = last_pos_acts.topk(10)

print("Top 10 firing features at position 39:")
for v, i in zip(values.tolist(), indices.tolist()):
    print(f"  feature {i:5d}: activation = {v:.3f}")

Top 10 firing features at position 39:
  feature  6315: activation = 3.827
  feature  8528: activation = 0.672
  feature 11197: activation = 0.471
  feature  6683: activation = 0.326
  feature  5610: activation = 0.219
  feature 23581: activation = 0.143
  feature 18330: activation = 0.019
  feature 16148: activation = 0.004
  feature     0: activation = 0.000
  feature     1: activation = 0.000


PHASE 2 : DRIVE Setup 

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
import os

PROJECT_DIR = "/content/drive/MyDrive/pgm-final"

os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/activations", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)

print("Project dir contents:")
for item in sorted(os.listdir(PROJECT_DIR)):
    print(" ", item)

Project dir contents:
  activations
  data
  results


Creating 5000 prompts

In [22]:
import torch
import numpy as np

N_PROMPTS = 5000
SEQ_LEN = 20
TOKEN_LOW = 1000
TOKEN_HIGH = 10000

torch.manual_seed(0)

random_tokens = torch.randint(
    low=TOKEN_LOW,
    high=TOKEN_HIGH,
    size=(N_PROMPTS, SEQ_LEN),
)

prompts = torch.cat([random_tokens, random_tokens], dim=1)

print("prompts shape:", prompts.shape)
print("prompts dtype:", prompts.dtype)
print("memory size:", prompts.element_size() * prompts.nelement() / 1e6, "MB")

prompts shape: torch.Size([5000, 40])
prompts dtype: torch.int64
memory size: 1.6 MB


In [23]:
PROMPTS_PATH = f"{PROJECT_DIR}/data/prompts.npy"
np.save(PROMPTS_PATH, prompts.numpy())
print("Saved to:", PROMPTS_PATH)

loaded = np.load(PROMPTS_PATH)
print("Loaded shape:", loaded.shape)
print("Match original:", np.array_equal(loaded, prompts.numpy()))

Saved to: /content/drive/MyDrive/pgm-final/data/prompts.npy
Loaded shape: (5000, 40)
Match original: True


In [26]:
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory

directory = get_pretrained_saes_directory()
release_info = directory["gpt2-small-res-jb"]

print("Available SAE IDs in gpt2-small-res-jb:")
for sae_id in release_info.saes_map.keys():
    print(" ", sae_id)

Available SAE IDs in gpt2-small-res-jb:
  blocks.0.hook_resid_pre
  blocks.1.hook_resid_pre
  blocks.2.hook_resid_pre
  blocks.3.hook_resid_pre
  blocks.4.hook_resid_pre
  blocks.5.hook_resid_pre
  blocks.6.hook_resid_pre
  blocks.7.hook_resid_pre
  blocks.8.hook_resid_pre
  blocks.9.hook_resid_pre
  blocks.10.hook_resid_pre
  blocks.11.hook_resid_pre
  blocks.11.hook_resid_post


In [27]:
import torch
torch.cuda.empty_cache()

from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained_no_processing(
    "gpt2",
    **sae.cfg.metadata.model_from_pretrained_kwargs,
)
model.eval()
print("Model reloaded with no-processing")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
Model reloaded with no-processing


In [28]:
del sae
torch.cuda.empty_cache()

from sae_lens import SAE

sae_L1 = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.1.hook_resid_pre",
    device="cuda",
)

sae_L2 = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.2.hook_resid_pre",
    device="cuda",
)

print("L1 SAE hook:", sae_L1.cfg.metadata.hook_name)
print("L2 SAE hook:", sae_L2.cfg.metadata.hook_name)
print("L1 d_sae:", sae_L1.cfg.d_sae)
print("L2 d_sae:", sae_L2.cfg.d_sae)

cfg.json: 0.00B [00:00, ?B/s]

blocks.1.hook_resid_pre/sae_weights.safe(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

blocks.1.hook_resid_pre/sparsity.safeten(…):   0%|          | 0.00/98.4k [00:00<?, ?B/s]

cfg.json: 0.00B [00:00, ?B/s]

blocks.2.hook_resid_pre/sae_weights.safe(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

blocks.2.hook_resid_pre/sparsity.safeten(…):   0%|          | 0.00/98.4k [00:00<?, ?B/s]

L1 SAE hook: blocks.1.hook_resid_pre
L2 SAE hook: blocks.2.hook_resid_pre
L1 d_sae: 24576
L2 d_sae: 24576


In [29]:
import numpy as np

prompts = np.load(f"{PROJECT_DIR}/data/prompts.npy")
prompts = torch.from_numpy(prompts).to("cuda")
print("prompts shape:", prompts.shape, "device:", prompts.device)

N_PROMPTS = prompts.shape[0]
BATCH_SIZE = 32
TARGET_POSITION = 39
D_SAE = sae_L1.cfg.d_sae

L1_HOOK = "blocks.1.hook_resid_pre"
L2_HOOK = "blocks.2.hook_resid_pre"

features_L1 = np.zeros((N_PROMPTS, D_SAE), dtype=np.float32)
features_L2 = np.zeros((N_PROMPTS, D_SAE), dtype=np.float32)

print(f"Allocated output arrays: ({N_PROMPTS}, {D_SAE}) each")
print(f"Memory per array: {features_L1.nbytes / 1e9:.2f} GB")

prompts shape: torch.Size([5000, 40]) device: cuda:0
Allocated output arrays: (5000, 24576) each
Memory per array: 0.49 GB


In [30]:
from tqdm import tqdm

with torch.no_grad():
    for batch_start in tqdm(range(0, N_PROMPTS, BATCH_SIZE)):
        batch_end = min(batch_start + BATCH_SIZE, N_PROMPTS)
        batch = prompts[batch_start:batch_end]

        _, cache = model.run_with_cache(
            batch,
            names_filter=[L1_HOOK, L2_HOOK],
        )

        resid_L1 = cache[L1_HOOK][:, TARGET_POSITION, :]
        resid_L2 = cache[L2_HOOK][:, TARGET_POSITION, :]

        feats_L1 = sae_L1.encode(resid_L1)
        feats_L2 = sae_L2.encode(resid_L2)

        features_L1[batch_start:batch_end] = feats_L1.cpu().numpy()
        features_L2[batch_start:batch_end] = feats_L2.cpu().numpy()

        del cache

100%|██████████| 157/157 [00:25<00:00,  6.11it/s]


In [31]:
L1_PATH = f"{PROJECT_DIR}/activations/features_L1_pos39.npy"
L2_PATH = f"{PROJECT_DIR}/activations/features_L2_pos39.npy"

np.save(L1_PATH, features_L1)
np.save(L2_PATH, features_L2)

print("Saved:")
print(" ", L1_PATH)
print(" ", L2_PATH)

print("\nSanity check — fraction non-zero (averaged across prompts):")
print("  L1:", (features_L1 > 0).mean())
print("  L2:", (features_L2 > 0).mean())

print("\nFeatures alive on >5% of prompts:")
print("  L1:", ((features_L1 > 0).mean(axis=0) > 0.05).sum())
print("  L2:", ((features_L2 > 0).mean(axis=0) > 0.05).sum())

Saved:
  /content/drive/MyDrive/pgm-final/activations/features_L1_pos39.npy
  /content/drive/MyDrive/pgm-final/activations/features_L2_pos39.npy

Sanity check — fraction non-zero (averaged across prompts):
  L1: 0.0009290283203125
  L2: 0.0012039225260416667

Features alive on >5% of prompts:
  L1: 32
  L2: 48
